In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_timeseries_overall;
CREATE TABLE dev.mohit_gangwani.ad_dma_timeseries_overall AS
-- Presence: distinct time bins per (ad_id, dma_name)
WITH presence AS (
  SELECT vc.external_id AS ad_id
  , vc.fk_dma_id
  , DATE_TRUNC('HOUR', vc.session_start) AS time_bin
  FROM prod.detection.viewing_commercials_firehose_dedup vc
  JOIN prod.detection.commercial_id_external_firehose cief
    ON cief.external_id = vc.external_id
  JOIN prod.detection.clients cl
    ON cl.client_id = cief.fk_client_id
  WHERE vc.session_start >= CURRENT_DATE - 7
    AND vc.session_start < CURRENT_DATE
    AND vc.fk_zoo_id = 17
    AND cl.client_name = 'kinetiq'
    AND vc.fk_dma_id IS NOT NULL
  GROUP BY 1, 2, 3
  HAVING COUNT(*) >= 10
),
-- bin counts
bins_per_dma AS (
  SELECT ad_id, fk_dma_id, COUNT(*) AS bin_count
  FROM presence
  GROUP BY 1, 2
),
-- Pairwise intersection counts (Jaccard numerator)
pair_intersections AS (
  SELECT p1.ad_id
  , p1.fk_dma_id AS dma_a
  , p2.fk_dma_id AS dma_b
  , COUNT(*) AS inter_bins
  FROM presence p1
  JOIN presence p2
    ON p1.ad_id = p2.ad_id
   AND p1.time_bin = p2.time_bin
   AND p1.fk_dma_id > p2.fk_dma_id
  GROUP BY 1,2,3
),
-- Bring in bin counts for union calculation
pair_with_union AS (
  SELECT i.ad_id
  , i.dma_a
  , i.dma_b
  , i.inter_bins
  , a.bin_count AS bins_a
  , b.bin_count AS bins_b
  , (a.bin_count + b.bin_count - i.inter_bins) AS union_bins
  FROM pair_intersections i
  JOIN bins_per_dma a
    ON i.ad_id = a.ad_id
   AND i.dma_a = a.fk_dma_id
  JOIN bins_per_dma b
    ON i.ad_id = b.ad_id
   AND i.dma_b = b.fk_dma_id
),
-- Pairwise Jaccard (co-coverage) per ad
pairwise_cocov AS (
  SELECT ad_id
  , dma_a
  , dma_b
  , CASE WHEN union_bins != 0 THEN inter_bins/union_bins END AS jaccard
  FROM pair_with_union
),
-- Mean co-coverage per ad across all DMA pairs
mean_cocoverage AS (
  SELECT ad_id
  , AVG(jaccard) AS mean_cocoverage
  , COUNT(*) AS num_dma_pairs
  FROM pairwise_cocov
  GROUP BY ad_id
)
SELECT m.ad_id
, m.mean_cocoverage
, m.num_dma_pairs
FROM mean_cocoverage m
;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dma_timeseries_overall
WHERE ad_id IN ('AE16546-2025-42-01766', 'AE16546-2025-40-02214', 'AE16546-2025-40-04118', 'AE16546-2025-36-00396', 'AE16546-2025-28-06509', 'AE16546-2025-27-01762', 'AE16546-2025-40-07328', 'AE16546-2025-40-01688', 'AE17151-2025-18-00132', 'AE16546-2025-43-00577')

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dma_timeseries_overall
WHERE ad_id IN ('AE16546-2025-43-02576', 'AE16546-2025-42-02508', 'AE16546-2025-43-02335', 'AE16546-2025-05-07669', 'AE16546-2025-30-04769', 'AE16546-2025-43-03760', 'AE16546-2025-32-06856', 'AE16546-2025-42-15347', 'AE16546-2025-35-04512', 'AE16546-2025-40-06231')

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dma_timeseries_overall
WHERE ad_id IN ('AE16546-2025-43-05892', 'AE16546-2024-45-01430', 'AE16546-2025-43-07032', 'AE16546-2025-43-05083', 'AE16546-2025-43-05730', 'AE16546-2025-43-04818', 'AE16546-2025-41-03971', 'AE16546-2023-42-05587', 'AE16546-2025-09-01406', 'AE16546-2024-42-01939')

In [0]:
mccdf = spark.sql("SELECT * FROM dev.mohit_gangwani.ad_dma_timeseries_overall").toPandas()
mccdf.head(10)

In [0]:
df = spark.sql("SELECT * FROM dev.mohit_gangwani.ad_dma_overall;").toPandas()

In [0]:
from calc_functions import *

In [0]:
agg_df = df.groupby(['ad_id']).agg(
    gini=('impression_count', lambda x: gini(x)),
    entropy=('impression_count', lambda x: locality_index(x))
).reset_index()
agg_df.describe()

In [0]:
final_df = pd.merge(agg_df, mccdf, on="ad_id", how='left')
final_df.describe()

In [0]:
final_df.notna().sum()

In [0]:
final_df.fillna(0, inplace=True)

In [0]:
scatterplot(df=final_df,
            x='gini',
            y='mean_cocoverage',
            title="Ad Footprint Concentration",
            xlabel="Normalized Gini (Inequality)",
            ylabel="Mean Co-Coverage")

In [0]:
scatterplot(df=final_df,
            x='entropy',
            y='mean_cocoverage',
            title="Ad Footprint Concentration",
            xlabel="Locality Index (Normalized Entropy)",
            ylabel="Mean Co-Coverage")

In [0]:
sns.scatterplot(
    data=final_df, 
    x='entropy', 
    y='mean_cocoverage',
    alpha=0.35,
    s=5
)
plt.title("Ad Footprint Concentration")
plt.xlabel("Locality Index (Normalized Entropy)")
plt.ylabel("Mean Co-Coverage")
plt.show()

In [0]:
sns.kdeplot(
    data=final_df,
    x='gini',
    y='mean_cocoverage',
    fill=True, alpha=0.4
)
plt.show()

In [0]:
sns.kdeplot(
    data=final_df,
    x='entropy', 
    y='mean_cocoverage',
    fill=True, alpha=0.4
)
plt.show()